# Total Energy

This notebook demonstrates how to run a workflow calculation using the Mat3ra API.

## Process Overview
1. Set up environment and parameters
2. Authenticate and initialize API client
3. Select account and project
4. Load and save materials
5. Configure workflow
6. Set up compute resources
7. Create and submit job
8. Monitor execution
9. Retrieve and visualize results

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
import sys

if sys.platform == "emscripten":
    import micropip

    await micropip.install("mat3ra-api-examples", deps=False)
    await micropip.install("mat3ra-utils")
    from mat3ra.utils.jupyterlite.packages import install_packages

    await install_packages("api_examples")


### 1.1. Set parameters and configurations for the workflow and job

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

# 2. Auth and organization parameters
# Set organization name to use it as the owner, otherwise your personal account is used
ORGANIZATION_NAME = None

# 3. Material parameters
FOLDER = "../uploads"
MATERIAL_NAME = "Silicon"  # Name of the material to load from local file or Standata

# 4. Workflow parameters
WORKFLOW_SEARCH_TERM = "total_energy.json"
MY_WORKFLOW_NAME = "Total Energy"
APPLICATION_NAME = "espresso"  # Specify application name (e.g., "espresso", "vasp", "nwchem")
ADD_RELAXATION = False  # Whether to add relaxation subworkflow
SAVE_WF_TO_COLLECTION = True  # If True, workflow is saved to collection

# 5. Compute parameters
CLUSTER_NAME = None  # specify i.e. "cluster-001" to use that cluster
QUEUE_NAME = QueueName.D
PPN = 1

# 6. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
JOB_NAME = f"Total Energy {timestamp}"
POLL_INTERVAL = 30  # seconds

### 1.2. API Configuration (optional, for local development)

## 2. Authenticate and initialize API client
### 2.1. Authenticate

In [ ]:
from utils.auth import authenticate

# Authenticate in the browser and have credentials stored in environment variables
await authenticate()

### 2.2. Initialize API Client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account to work under

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Create material
### 3.1. Load material from local file (or Standata)

In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.materials import Materials
from utils.visualize import visualize_materials as visualize
from utils.jupyterlite import load_material_from_folder

material = load_material_from_folder(FOLDER, MATERIAL_NAME) or Material.create(
    Materials.get_by_name_first_match(MATERIAL_NAME))

visualize(material)

### 3.2. Save material to the platform

In [ ]:
from utils.api import get_or_create_material
from utils.generic import dict_to_namespace

saved_material_response = get_or_create_material(client.materials, material, ACCOUNT_ID)
saved_material = dict_to_namespace(saved_material_response)

## 4. Create workflow and set its parameters
### 4.1. Get list of applications and select one

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.ade.application import Application

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)
print(f"Using application: {app.name}")

### 4.2. Create workflow from standard workflows and preview it

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from utils.visualize import visualize_workflow

workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(WORKFLOW_SEARCH_TERM)
workflow = Workflow.create(workflow_config)
workflow.name = MY_WORKFLOW_NAME

visualize_workflow(workflow)

### 4.3. Modify workflow (Optional)

In [ ]:
# Example: Add relaxation subworkflow
if ADD_RELAXATION:
    workflow.add_relaxation()


# # Example: Change model parameters
# from mat3ra.mode import Model
# from mat3ra.standata.model_tree import ModelTreeStandata
#
# model_config = ModelTreeStandata.get_model_by_parameters(
#     type="dft",
#     subtype="gga",
#     functional="pbe"
# )
# model_config["method"] = {"type": "pseudopotential", "subtype": "us"}
# model = Model.create(model_config)
#
# for subworkflow in workflow.subworkflows:
#     subworkflow.model = model

# # Example: Modify k-grids
# from mat3ra.wode.context.providers import PointsGridDataProvider
#
# new_context = PointsGridDataProvider(dimensions=KGRID, isEdited=True).yield_data()
# subworkflow = workflow.subworkflows[0]
# unit = subworkflow.get_unit_by_name(name="pw_scf")  # Adjust unit name as needed
# unit.add_context(new_context)
# subworkflow.set_unit(unit)
#
# # Preview modified workflow
# visualize_workflow(workflow)

### 4.4. Save workflow to collection

In [ ]:
from utils.generic import dict_to_namespace

workflow_id_or_dict = None

if SAVE_WF_TO_COLLECTION:
    from utils.api import get_or_create_workflow

    saved_workflow_response = get_or_create_workflow(client.workflows, workflow, ACCOUNT_ID)
    saved_workflow = dict_to_namespace(saved_workflow_response)
    workflow_id_or_dict = saved_workflow._id
    print(f"Workflow ID: {saved_workflow._id}")
else:
    workflow_id_or_dict = workflow.to_dict()
    print("Workflow will be embedded into job (not saved to collection)")

## 5. Create the compute configuration
### 5.1. Get list of clusters

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c for c in clusters]}")

### 5.2. Create compute configuration for the job


In [ ]:
from mat3ra.ide.compute import Compute

# Select first available cluster or use specified name
cluster = next((c for c in clusters if c["hostname"] == CLUSTER_NAME), clusters[0] if clusters else None)

compute = Compute(
    cluster=cluster,
    queue=QUEUE_NAME,
    ppn=PPN
)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")

## 6. Create the job with material and workflow configuration
### 6.1. Create job

In [ ]:
from utils.api import create_job
from utils.visualize import display_JSON

print(f"Material: {saved_material._id}")
print(f"Workflow: {workflow_id_or_dict if SAVE_WF_TO_COLLECTION else '(embedded)'}")
print(f"Project: {project_id}")

job_response = create_job(
    jobs_endpoint=client.jobs,
    materials=[vars(saved_material)],
    workflow_id_or_dict=workflow_id_or_dict,
    project_id=project_id,
    owner_id=ACCOUNT_ID,
    prefix=JOB_NAME,
    compute=compute.to_dict(),
    save_to_collection=SAVE_WF_TO_COLLECTION,
)

job_dict = job_response[0]
job = dict_to_namespace(job_dict)
job_id = job._id
print("✅ Job created successfully!")
print(f"Job ID: {job_id}")
display_JSON(job_response)

## 7. Submit the job and monitor the status

In [ ]:
client.jobs.submit(job_id)
print(f"✅ Job {job_id} submitted successfully!")

In [ ]:
from utils.api import wait_for_jobs_to_finish_async

await wait_for_jobs_to_finish_async(client.jobs, [job_id], poll_interval=POLL_INTERVAL)

## 8. Retrieve results

In [ ]:
from mat3ra.prode import PropertyName
from utils.visualize import visualize_properties

property_data = client.properties.get_for_job(job_id, property_name=PropertyName.scalar.total_energy.value)
visualize_properties(property_data, title="Total Energy")
